In [ ]:
#!/usr/bin/env python3
"""
Create histograms of Turbine CAPEX and total project CAPEX
across all scenario result files.

Assumptions:
- scenarios.json exists in the RESULTS_FOLDER and contains a list of scenarios,
  each with a "scenario_id".
- For each scenario_id SID there is a parquet file named:
      {TABLE_NAME}_df_{SID}.parquet
- Each parquet file is assumed to contain CAPEX cost_records-like data with
  at least the columns: "cost" (numeric), "turbine_id" (can be NaN for non-turbine rows).

Outputs:
- Histograms saved as PNG in RESULTS_FOLDER:
    - histogram_total_capex.png
    - histogram_turbine_capex.png
"""

from pathlib import Path
import json
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# 1) Configuration
# --------------------------------------------------------------------
RESULTS_FOLDER = r"M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty"
TABLE_NAME = "capex"  # adapt to your actual table name, e.g. "capex_cost_records"

RESULTS_FOLDER = Path(RESULTS_FOLDER)
assert RESULTS_FOLDER.exists(), f"Results folder not found: {RESULTS_FOLDER}"

SCENARIOS_PATH = RESULTS_FOLDER / "scenarios.json"
assert SCENARIOS_PATH.exists(), f"scenarios.json not found at: {SCENARIOS_PATH}"

# --------------------------------------------------------------------
# 2) Load scenarios.json
# --------------------------------------------------------------------
with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

if isinstance(scenarios, dict) and "scenarios" in scenarios:
    scenarios = scenarios["scenarios"]

assert isinstance(scenarios, list), "Expected a list of scenarios in scenarios.json"
print(f"Loaded {len(scenarios)} scenarios from {SCENARIOS_PATH}")

# --------------------------------------------------------------------
# 3) Load parquet result DataFrames
# --------------------------------------------------------------------
RESULTS: Dict[str, pd.DataFrame] = {}
missing = []

for sc in scenarios:
    sid = str(sc.get("scenario_id", "")).strip()
    if not sid:
        continue

    parquet_path = RESULTS_FOLDER / f"{TABLE_NAME}_df_{sid}.parquet"
    if parquet_path.exists():
        try:
            df = pd.read_parquet(parquet_path)
            RESULTS[sid] = df
        except Exception as e:
            missing.append((sid, str(parquet_path), f"Read error: {e}"))
    else:
        missing.append((sid, str(parquet_path), "Missing file"))

print(f"Loaded {len(RESULTS)} DataFrame(s).")
if missing:
    print("\nFiles not loaded:")
    for m in missing:
        print(" - scenario_id=", m[0], " path=", m[1], " reason=", m[2])

if not RESULTS:
    raise RuntimeError("No result DataFrames loaded; nothing to plot.")

# --------------------------------------------------------------------
# 4) Compute total CAPEX and turbine CAPEX
# --------------------------------------------------------------------
total_capex_per_scenario: List[float] = []
turbine_capex_all: List[float] = []

for sid, df in RESULTS.items():
    # ensure cost column is numeric
    if "cost" not in df.columns:
        print(f"WARNING: scenario {sid} has no 'cost' column; skipping.")
        continue

    df = df.copy()
    df["cost"] = pd.to_numeric(df["cost"], errors="coerce").fillna(0.0)

    # total project CAPEX (all rows)
    total_capex = float(df["cost"].sum())
    total_capex_per_scenario.append(total_capex)

    # per-turbine CAPEX (sum of cost over rows assigned to a turbine_id)
    if "turbine_id" in df.columns:
        # treat any non-NaN turbine_id as turbine row
        df_turb = df[df["turbine_id"].notna()].copy()
        if not df_turb.empty:
            # group by turbine_id and sum cost
            capex_by_turbine = (
                df_turb.groupby("turbine_id", dropna=True)["cost"]
                .sum()
                .astype(float)
            )
            turbine_capex_all.extend(capex_by_turbine.values.tolist())
    else:
        print(f"WARNING: scenario {sid} has no 'turbine_id' column; skipping turbine CAPEX.")

print(f"\nComputed total CAPEX for {len(total_capex_per_scenario)} scenario(s).")
print(f"Collected {len(turbine_capex_all)} turbine CAPEX values across all scenarios.")

if not total_capex_per_scenario:
    raise RuntimeError("No total CAPEX values computed; cannot plot histogram.")

if not turbine_capex_all:
    print("WARNING: No turbine CAPEX values collected; turbine histogram will not be created.")

# --------------------------------------------------------------------
# 5) Create histograms
# --------------------------------------------------------------------
# Helper to format axis labels in millions if values are large
def _maybe_millions(ax, values, label: str):
    vmax = np.max(values)
    if vmax >= 1e6:
        scaled = np.array(values) / 1e6
        ax.set_xlabel(label + " [million]")
        return scaled
    else:
        ax.set_xlabel(label)
        return np.array(values)

# 5a) Total CAPEX histogram
plt.figure(figsize=(8, 5))
vals_total = _maybe_millions(plt.gca(), total_capex_per_scenario, "Total project CAPEX")
plt.hist(vals_total, bins=30, edgecolor="black", alpha=0.75)
plt.ylabel("Frequency")
plt.title("Histogram of Total Project CAPEX Across Scenarios")
plt.grid(True, linestyle="--", alpha=0.3)
out_path_total = RESULTS_FOLDER / "histogram_total_capex.png"
plt.tight_layout()
plt.savefig(out_path_total, dpi=150)
plt.close()
print(f"Saved total CAPEX histogram to: {out_path_total}")

# 5b) Turbine CAPEX histogram (if available)
if turbine_capex_all:
    plt.figure(figsize=(8, 5))
    vals_turb = _maybe_millions(plt.gca(), turbine_capex_all, "Per-turbine CAPEX")
    plt.hist(vals_turb, bins=30, edgecolor="black", alpha=0.75)
    plt.ylabel("Frequency")
    plt.title("Histogram of Turbine CAPEX Across All Scenarios")
    plt.grid(True, linestyle="--", alpha=0.3)
    out_path_turb = RESULTS_FOLDER / "histogram_turbine_capex.png"
    plt.tight_layout()
    plt.savefig(out_path_turb, dpi=150)
    plt.close()
    print(f"Saved turbine CAPEX histogram to: {out_path_turb}")

print("\nDone.")


Loaded 100 scenarios from M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty\scenarios.json
Loaded 100 DataFrame(s).

Computed total CAPEX for 100 scenario(s).
Collected 3400 turbine CAPEX values across all scenarios.
Saved total CAPEX histogram to: M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty\histogram_total_capex.png
Saved turbine CAPEX histogram to: M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty\histogram_turbine_capex.png

Done.


: 

#### Scenario Analalysis

In [9]:
#!/usr/bin/env python3
"""
Create histograms of Turbine CAPEX and total project CAPEX
for each macro scenario (Scenario.name) and combined across scenarios.

Definitions:
- "Macro scenario" = Scenario.name (e.g. "Open Door", "Increased Barriers", ...)
- Each macro scenario has multiple replicates; each replicate is a scenario entry
  with its own scenario_id and parquet file.

Assumptions:
- scenarios.json exists in the RESULTS_FOLDER and contains a list of scenarios
  with at least:
      - "scenario_id"
      - "overrides" dict that includes "Scenario.name"
- For each scenario_id SID there is a parquet file named:
      {TABLE_NAME}_df_{SID}.parquet
- Each parquet file contains CAPEX records with at least:
      - "cost" (numeric)
      - "turbine_id" (NaN for non-turbine rows)

For each macro scenario S:
- We compute:
    - total project CAPEX per replicate (sum of all costs in that file)
    - per-turbine CAPEX per replicate using a single reference turbine
      (turbine_id == 1 if present, otherwise the first turbine)

Outputs (in RESULTS_FOLDER):
- Per macro scenario:
    - histogram_total_capex_<ScenarioNameSanitized>.png
    - histogram_turbine_capex_<ScenarioNameSanitized>.png
- Combined (all macro scenarios overlaid, colored with legend):
    - histogram_total_capex_all_scenarios.png
    - histogram_turbine_capex_all_scenarios.png
"""

from pathlib import Path
import json
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# 1) Configuration
# --------------------------------------------------------------------
RESULTS_FOLDER = r"M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty"
TABLE_NAME = "capex"  # prefix of parquet files: capex_df_<scenario_id>.parquet

RESULTS_FOLDER = Path(RESULTS_FOLDER)
assert RESULTS_FOLDER.exists(), f"Results folder not found: {RESULTS_FOLDER}"

SCENARIOS_PATH = RESULTS_FOLDER / "scenarios.json"
assert SCENARIOS_PATH.exists(), f"scenarios.json not found at: {SCENARIOS_PATH}"

# --------------------------------------------------------------------
# 2) Load scenarios.json
# --------------------------------------------------------------------
with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

# Might be {"scenarios": [...]} or just [...]
if isinstance(scenarios, dict) and "scenarios" in scenarios:
    scenarios = scenarios["scenarios"]

assert isinstance(scenarios, list), "Expected a list of scenarios in scenarios.json"
print(f"Loaded {len(scenarios)} scenario entries from {SCENARIOS_PATH}")

# --------------------------------------------------------------------
# 3) Helpers
# --------------------------------------------------------------------
def get_macro_name(sc: Dict[str, Any], default: str) -> str:
    """
    Extract the macro scenario name from a scenario entry.

    Priority:
    1. sc["overrides"]["Scenario.name"]  (if present)
    2. sc.get("Scenario.name")
    3. sc.get("label")
    4. fallback: default (e.g. scenario_id)
    """
    overrides = sc.get("overrides") or {}
    name = overrides.get("Scenario.name") or sc.get("Scenario.name") \
           or sc.get("label") or default
    return str(name)


def _sanitize_name(name: str) -> str:
    """Make a string safe for filenames."""
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in name)


def _maybe_millions_single(ax, values: np.ndarray, label: str) -> np.ndarray:
    """Scale a single array to millions if needed, and set x-axis label."""
    if values.size == 0:
        ax.set_xlabel(label)
        return values
    vmax = float(np.max(values))
    if vmax >= 1e6:
        ax.set_xlabel(label + " [million]")
        return values / 1e6
    else:
        ax.set_xlabel(label)
        return values


def _maybe_millions_multi(ax, arrays: List[np.ndarray], label: str) -> List[np.ndarray]:
    """Scale multiple arrays jointly based on global max."""
    nonempty = [arr for arr in arrays if arr.size > 0]
    if not nonempty:
        ax.set_xlabel(label)
        return arrays
    all_vals = np.concatenate(nonempty)
    vmax = float(np.max(all_vals))
    if vmax >= 1e6:
        ax.set_xlabel(label + " [million]")
        return [arr / 1e6 for arr in arrays]
    else:
        ax.set_xlabel(label)
        return arrays

# --------------------------------------------------------------------
# 4) Group scenario_ids by macro scenario name
# --------------------------------------------------------------------
macro_to_entries: Dict[str, Dict[str, Any]] = {}  # macro_name -> sid -> scenario_entry

for sc in scenarios:
    sid = str(sc.get("scenario_id", "")).strip()
    if not sid:
        continue
    macro_name = get_macro_name(sc, default=sid)
    # deduplicate by scenario_id within each macro scenario
    macro_to_entries.setdefault(macro_name, {})
    macro_to_entries[macro_name][sid] = sc

print("\nMacro scenarios found (unique scenario_ids):")
for macro, entries_by_sid in macro_to_entries.items():
    print(f"  {macro}: {len(entries_by_sid)} replicate(s)")

if not macro_to_entries:
    raise RuntimeError("No macro scenarios could be identified; check Scenario.name in scenarios.json")

# --------------------------------------------------------------------
# 5) Load parquet data and aggregate per macro scenario
# --------------------------------------------------------------------
macro_data: Dict[str, Dict[str, Any]] = {}
missing_files: List[Any] = []

for macro_name, entries_by_sid in macro_to_entries.items():
    total_capex_list: List[float] = []   # one value per replicate
    turbine_capex_list: List[float] = [] # one value per replicate (reference turbine)

    for sid, sc in entries_by_sid.items():
        parquet_path = RESULTS_FOLDER / f"{TABLE_NAME}_df_{sid}.parquet"
        if not parquet_path.exists():
            missing_files.append((macro_name, sid, str(parquet_path), "Missing file"))
            continue

        try:
            df = pd.read_parquet(parquet_path)
        except Exception as e:
            missing_files.append((macro_name, sid, str(parquet_path), f"Read error: {e}"))
            continue

        if "cost" not in df.columns:
            print(f"WARNING: macro '{macro_name}', scenario_id {sid} has no 'cost' column; skipping this file.")
            continue

        df = df.copy()
        df["cost"] = pd.to_numeric(df["cost"], errors="coerce").fillna(0.0)

        # ---- total project CAPEX for this replicate ----
        total_capex_list.append(float(df["cost"].sum()))

        # ---- per-turbine CAPEX: use ONLY turbine 1 (or first turbine) as representative ----
        if "turbine_id" in df.columns:
            df_turb = df[df["turbine_id"].notna()].copy()
            if not df_turb.empty:
                # group by turbine_id
                capex_by_turbine = (
                    df_turb.groupby("turbine_id", dropna=True)["cost"]
                    .sum()
                    .astype(float)
                )

                idx = capex_by_turbine.index
                ref_value = None

                # case 1: turbine ids are numeric and contain 1
                if 1 in idx:
                    ref_value = capex_by_turbine.loc[1]
                # case 2: turbine ids are string "1"
                elif "1" in idx:
                    ref_value = capex_by_turbine.loc["1"]
                else:
                    # fallback: just take the first turbine in sorted order
                    ref_value = capex_by_turbine.sort_index().iloc[0]

                turbine_capex_list.append(float(ref_value))
        else:
            print(
                f"WARNING: macro '{macro_name}', scenario_id {sid} has no 'turbine_id' column; "
                "turbine CAPEX skipped for this file."
            )

    total_capex_arr = np.array(total_capex_list, dtype=float)
    turbine_capex_arr = np.array(turbine_capex_list, dtype=float)

    if total_capex_arr.size or turbine_capex_arr.size:
        macro_data[macro_name] = {
            "total_capex": total_capex_arr,
            "turbine_capex": turbine_capex_arr,
        }

        # Debug print: how many samples are actually used?
        print(
            f"\nMacro scenario '{macro_name}': "
            f"{total_capex_arr.size} replicate total-CAPEX values, "
            f"{turbine_capex_arr.size} per-turbine CAPEX values"
        )

print(f"\nCollected CAPEX data for {len(macro_data)} macro scenario(s).")
if missing_files:
    print("\nFiles not loaded:")
    for m in missing_files:
        print(f" - macro='{m[0]}', scenario_id={m[1]}, path={m[2]}, reason={m[3]}")

if not macro_data:
    raise RuntimeError("No usable macro scenario data; cannot plot.")

# --------------------------------------------------------------------
# 6) Per-macro-scenario histograms (with Gaussian curve fit)
# --------------------------------------------------------------------
print("\nCreating per-macro-scenario histograms...")

for macro_name, data in macro_data.items():
    total_capex_arr = data["total_capex"]
    turbine_capex_arr = data["turbine_capex"]

    safe_name = _sanitize_name(macro_name)

    # 6a) Histogram of total project CAPEX across replicates
    if total_capex_arr.size > 0:
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        vals = _maybe_millions_single(ax, total_capex_arr, "Total project CAPEX")

        bins = min(30, max(1, vals.size))

        # histogram
        n, bin_edges, _ = ax.hist(
            vals,
            bins=bins,
            edgecolor="C0",
            alpha=0.6,
            linewidth=1.2,
            label="Histogram",
        )

        # Gaussian fit
        if vals.size > 1:
            mu = float(vals.mean())
            sigma = float(vals.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * vals.size
                    * bin_width
                )
                ax.plot(x, y, color="C0", linewidth=1.8, label="Gaussian fit")

        ax.set_ylabel("Frequency")
        ax.set_title(f"Total project CAPEX\nScenario: {macro_name}")
        ax.grid(True, linestyle="--", alpha=0.3)
        ax.legend()
        out_path = RESULTS_FOLDER / f"histogram_total_capex_{safe_name}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"  Saved total CAPEX histogram for '{macro_name}' to: {out_path}")

    # 6b) Histogram of per-turbine CAPEX across replicates
    if turbine_capex_arr.size > 0:
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        vals = _maybe_millions_single(ax, turbine_capex_arr, "Per-turbine CAPEX")

        bins = min(30, max(1, vals.size))

        # histogram
        n, bin_edges, _ = ax.hist(
            vals,
            bins=bins,
            edgecolor="C1",
            alpha=0.6,
            linewidth=1.2,
            label="Histogram",
        )

        # Gaussian fit
        if vals.size > 1:
            mu = float(vals.mean())
            sigma = float(vals.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * vals.size
                    * bin_width
                )
                ax.plot(x, y, color="C1", linewidth=1.8, label="Gaussian fit")

        ax.set_ylabel("Frequency")
        ax.set_title(f"Per-turbine CAPEX\nScenario: {macro_name}")
        ax.grid(True, linestyle="--", alpha=0.3)
        ax.legend()
        out_path = RESULTS_FOLDER / f"histogram_turbine_capex_{safe_name}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"  Saved turbine CAPEX histogram for '{macro_name}' to: {out_path}")

# --------------------------------------------------------------------
# 7) Combined histograms (all macro scenarios overlaid, with fits)
# --------------------------------------------------------------------
print("\nCreating combined histograms for all macro scenarios...")

macro_names = list(macro_data.keys())
total_arrays = [macro_data[m]["total_capex"] for m in macro_names]
turbine_arrays = [macro_data[m]["turbine_capex"] for m in macro_names]

# use matplotlib default color cycle
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# 7a) Combined total project CAPEX histogram
nonempty_total = [a for a in total_arrays if a.size > 0]
if nonempty_total:
    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    scaled_total_arrays = _maybe_millions_multi(ax, total_arrays, "Total project CAPEX")

    nonempty_scaled = [a for a in scaled_total_arrays if a.size > 0]
    all_vals = np.concatenate(nonempty_scaled)
    bins = min(30, max(5, len(all_vals)))
    bin_edges = np.linspace(all_vals.min(), all_vals.max(), bins)

    for i, (macro_name, arr) in enumerate(zip(macro_names, scaled_total_arrays)):
        if arr.size == 0:
            continue

        color = color_cycle[i % len(color_cycle)]

        # histogram (step)
        n, _, _ = ax.hist(
            arr,
            bins=bin_edges,
            histtype="step",
            linewidth=1.5,
            alpha=0.9,
            label=macro_name,
            color=color,
        )

        # Gaussian fit
        if arr.size > 1:
            mu = float(arr.mean())
            sigma = float(arr.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * arr.size
                    * bin_width
                )
                ax.plot(x, y, color=color, linestyle="-", linewidth=1.5)

    ax.set_ylabel("Frequency")
    ax.set_title("Total project CAPEX\n(all macro scenarios)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    out_path = RESULTS_FOLDER / "histogram_total_capex_all_scenarios.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved combined total CAPEX histogram to: {out_path}")
else:
    print("No total project CAPEX data available; skipping combined total CAPEX plot.")

# 7b) Combined per-turbine CAPEX histogram
nonempty_turb = [a for a in turbine_arrays if a.size > 0]
if nonempty_turb:
    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    scaled_turb_arrays = _maybe_millions_multi(ax, turbine_arrays, "Per-turbine CAPEX")

    nonempty_scaled_turb = [a for a in scaled_turb_arrays if a.size > 0]
    all_vals_turb = np.concatenate(nonempty_scaled_turb)
    bins_turb = 30
    bin_edges_turb = np.linspace(all_vals_turb.min(), all_vals_turb.max(), bins_turb)

    for i, (macro_name, arr) in enumerate(zip(macro_names, scaled_turb_arrays)):
        if arr.size == 0:
            continue

        color = color_cycle[i % len(color_cycle)]

        # histogram (step)
        n, _, _ = ax.hist(
            arr,
            bins=bin_edges_turb,
            histtype="step",
            linewidth=1.5,
            alpha=0.9,
            label=macro_name,
            color=color,
        )

        # Gaussian fit
        if arr.size > 1:
            mu = float(arr.mean())
            sigma = float(arr.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges_turb[0], bin_edges_turb[-1], 200)
                bin_width = bin_edges_turb[1] - bin_edges_turb[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * arr.size
                    * bin_width
                )
                ax.plot(x, y, color=color, linestyle="-", linewidth=1.5)

    ax.set_ylabel("Frequency")
    ax.set_title("Per-turbine CAPEX\n(all macro scenarios)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    out_path = RESULTS_FOLDER / "histogram_turbine_capex_all_scenarios.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved combined turbine CAPEX histogram to: {out_path}")
else:
    print("No per-turbine CAPEX data available; skipping combined turbine plot.")

print("\nDone.")


Loaded 400 scenario entries from M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty\scenarios.json

Macro scenarios found (unique scenario_ids):
  Open Door: 100 replicate(s)
  Increased Barriers: 100 replicate(s)
  Economic Downturn: 100 replicate(s)
  Global Escalation: 100 replicate(s)

Macro scenario 'Open Door': 100 replicate total-CAPEX values, 100 per-turbine CAPEX values

Macro scenario 'Increased Barriers': 100 replicate total-CAPEX values, 100 per-turbine CAPEX values

Macro scenario 'Economic Downturn': 100 replicate total-CAPEX values, 100 per-turbine CAPEX values

Macro scenario 'Global Escalation': 100 replicate total-CAPEX values, 100 per-turbine CAPEX values

Collected CAPEX data for 4 macro scenario(s).

Creating per-macro-scenario histograms...
  Saved total CAPEX histogram for 'Open Door' to: M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\CAPEX_Uncertainty\histogram_total_capex_Open_Door.png
  